# Task 1 — Define Source and Extract (Sabbar)

## Source definition
- **Name:** Sabbar Jobs (sabbar.com)
- **Endpoint / access:** public job pages under `https://sabbar.com/en/jobs/...` (scraped with Playwright — no public API)
- **Authentication:** none
- **Method:** two stages — Stage 1 collects job URLs into `job_urls.csv`; Stage 2 (this notebook) opens each URL and extracts the embedded Next.js `jobDetails` object
- **Rate limits:** self-imposed — 3 concurrent pages, 3–6s random delay, 3 retries per URL
- **Licence / terms:** public listings; scraped for academic use only

**Contract:** reads `../data/raw/job_urls.csv`, writes raw output to `../data/raw/sabbar_<date>.json`. Nothing is modified in the raw output.

In [1]:
import asyncio
import csv
import json
import random
from datetime import date
from playwright.async_api import async_playwright

# ---- Configuration ----
INPUT_FILE  = "../data/raw/job_urls.csv"
OUTPUT_FILE = f"../data/raw/sabbar_{date.today()}.json"
FAILED_FILE = "../data/raw/failed_urls.csv"
LIMIT       = None      # None = scrape all. Set a number to test (e.g. 50).

CONCURRENCY = 3         # parallel pages
MIN_DELAY   = 3         # min seconds between requests
MAX_DELAY   = 6         # max seconds between requests
RETRIES     = 3         # retry attempts per URL
SAVE_EVERY  = 100       # auto-save progress every N jobs

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
)

ModuleNotFoundError: No module named 'playwright'

## Data extraction functions

In [ ]:
def extract_job_details(html):
    """Extract the jobDetails object embedded in the Next.js page data.
    The data is JS-escaped (\\" instead of "), so we unescape then parse.
    Returns a dict, or None if the object cannot be found/parsed."""
    marker = '\\"jobPositionValue\\"'
    pos = html.find(marker)
    if pos == -1:
        return None

    start = html.rfind('{\\"id\\":', 0, pos)
    if start == -1:
        start = html.rfind("{", 0, pos)

    depth, i, in_string = 0, start, False
    while i < len(html):
        if html[i] == "\\" and i + 1 < len(html) and html[i + 1] == '"':
            in_string = not in_string
            i += 2
            continue
        char = html[i]
        if not in_string:
            if char == "{":
                depth += 1
            elif char == "}":
                depth -= 1
                if depth == 0:
                    raw = html[start:i + 1]
                    clean = raw.replace('\\"', '"').replace("\\\\", "\\")
                    try:
                        return json.loads(clean)
                    except Exception:
                        return None
        i += 1
    return None


def pick_fields(details, url):
    """Select and organize the useful fields from the raw jobDetails object."""
    description = (details.get("description") or "")
    description = description.replace("<br>", "\n").replace("&nbsp;", " ").strip()

    return {
        "url":                  url,
        "id":                   details.get("id"),
        "title":                details.get("jobPositionValue"),
        "company":              (details.get("partnerName") or "").strip(),
        "industries":           ", ".join(details.get("industries") or []),
        "salary":               details.get("salary"),
        "max_salary":           details.get("maxSalary"),
        "currency":             details.get("currency"),
        "salary_frequency":     details.get("salaryFrequency"),
        "contract_type":        details.get("contractType"),
        "workplace_type":       details.get("workplaceType"),
        "experience_years":     details.get("experienceYears"),
        "requires_experience":  details.get("requiresExperience"),
        "requires_english":     details.get("requiresEnglish"),
        "english_proficiency":  details.get("englishProficiency"),
        "requires_gosi":        details.get("requiresGosi"),
        "requires_health_card": details.get("requiresHealthCard"),
        "gender":               details.get("gender"),
        "saudis_only":          details.get("saudisOnly"),
        "females_only_env":     details.get("isEnvironmentForFemalesOnly"),
        "city":                 details.get("cityValue"),
        "district":             details.get("district"),
        "address":              details.get("address"),
        "country":              details.get("countryValue"),
        "postal_code":          details.get("postalCode"),
        "created_date":         details.get("createdDate"),
        "job_status":           details.get("jobStatus"),
        "is_new":               details.get("isNew"),
        "source":               details.get("source"),
        "description":          description,
    }

## Scraper (pagination, retries, error handling)

In [ ]:
async def scrape_job(page, url):
    """Open a job page and extract its data, retrying on failure."""
    for attempt in range(RETRIES):
        try:
            await page.goto(url, wait_until="domcontentloaded", timeout=60000)
            html = await page.content()
            details = extract_job_details(html)
            return pick_fields(details, url) if details else None
        except Exception:
            if attempt < RETRIES - 1:
                await asyncio.sleep(3)
                continue
            return None


def save_progress(results):
    """Write the collected results to the output JSON file."""
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)


async def worker(context, queue, results, failed, counter, total, lock):
    """Pull URLs from the queue and scrape until the queue is empty."""
    page = await context.new_page()
    while True:
        try:
            url = queue.get_nowait()
        except asyncio.QueueEmpty:
            break

        job = await scrape_job(page, url)
        async with lock:
            counter[0] += 1
            position = counter[0]
            if job:
                results.append(job)
                print(f"[{position}/{total}] OK: {job['title']}")
                if len(results) % SAVE_EVERY == 0:
                    save_progress(results)
            else:
                failed.append(url)
                print(f"[{position}/{total}] FAILED")
        await asyncio.sleep(random.uniform(MIN_DELAY, MAX_DELAY))
    await page.close()

## Input / output helpers and main routine

In [ ]:
def load_urls():
    """Read the English job URLs from the input CSV file."""
    with open(INPUT_FILE, "r", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        next(reader)                        # skip header
        urls = [row[0] for row in reader]
    return urls[:LIMIT] if LIMIT else urls


def save_failed(failed):
    """Write failed URLs to a CSV file for later retry."""
    with open(FAILED_FILE, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerow(["url"])
        writer.writerows([u] for u in failed)


async def main():
    urls = load_urls()
    total = len(urls)
    print(f"Total URLs: {total}")

    queue = asyncio.Queue()
    for url in urls:
        queue.put_nowait(url)

    results, failed, counter, lock = [], [], [0], asyncio.Lock()

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(user_agent=USER_AGENT)
        workers = [
            asyncio.create_task(
                worker(context, queue, results, failed, counter, total, lock)
            )
            for _ in range(CONCURRENCY)
        ]
        await asyncio.gather(*workers)
        await browser.close()

    save_progress(results)
    save_failed(failed)
    print(f"\nDone. Saved {len(results)} jobs -> {OUTPUT_FILE}. Failed: {len(failed)}")

## Run the scrape

In a notebook the event loop is already running, so call `await main()` directly
(not `asyncio.run(...)`). Requires: `pip install playwright` then `playwright install chromium`.

In [ ]:
await main()

## Verify saved output

In [3]:
import json
with open("../data/raw/sabbar_2026-09-10.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("Records saved:", len(data))
print("Columns:", list(data[0].keys()))
data[0]

Records saved: 15678
Columns: ['url', 'id', 'title', 'company', 'industries', 'salary', 'max_salary', 'currency', 'salary_frequency', 'contract_type', 'workplace_type', 'experience_years', 'requires_experience', 'requires_english', 'english_proficiency', 'requires_gosi', 'requires_health_card', 'gender', 'saudis_only', 'females_only_env', 'city', 'district', 'address', 'country', 'postal_code', 'created_date', 'job_status', 'is_new', 'source', 'description']


{'url': 'https://sabbar.com/en/jobs/c-jeddah-r-%D8%A8%D8%A7%D8%A6%D8%B9-%D8%A3%D8%AF%D9%88%D8%A7%D8%AA-%D9%88%D8%A3%D8%AC%D9%87%D8%B2%D8%A9-%D9%85%D9%86%D8%B2%D9%84%D9%8A%D8%A9/id-e5e16dbc-d4d3-4529-bcf6-fe9e1d66f100',
 'id': 'e5e16dbc-d4d3-4529-bcf6-fe9e1d66f100',
 'title': 'بائع أدوات وأجهزة منزلية',
 'company': 'اكسترا',
 'industries': 'Retail',
 'salary': None,
 'max_salary': None,
 'currency': 'SAR',
 'salary_frequency': 'MONTHLY',
 'contract_type': 'PARTTIME',
 'workplace_type': 'REMOTE',
 'experience_years': '0-1',
 'requires_experience': False,
 'requires_english': False,
 'english_proficiency': None,
 'requires_gosi': False,
 'requires_health_card': False,
 'gender': 'FEMALE',
 'saudis_only': True,
 'females_only_env': False,
 'city': 'Jeddah',
 'district': '',
 'address': 'Jeddah',
 'country': 'Saudi Arabia',
 'postal_code': None,
 'created_date': 1788992940,
 'job_status': 'OPEN',
 'is_new': True,
 'source': 'IN_PLATFORM',
 'description': '<p>نبحث عن بائع متحمس وملتزم للانضم